# Lojistik Regresyon Algoritması (Logistic Regression)
### Banka Pazarlama Kampanyası ile Abone Olma Durumu Tahmini

---

**Veri Seti:** Bank Marketing Dataset — Müşterilerin vadeli mevduat hesabı açıp açmayacağının tahmini

---

## 1. Konu Anlatımı — Lojistik Regresyon Nedir?

Lojistik regresyon, **sınıflandırma** problemleri için kullanılan bir makine öğrenmesi algoritmasıdır. Adında "regresyon" geçmesine rağmen aslında bir sınıflandırma yöntemidir.

### 1.1. Temel Mantık
Lineer regresyonda çıktı sürekli bir değer iken (ör: ev fiyatı), lojistik regresyonda çıktı kategoriktir (ör: abone oldu: 1 / olmadı: 0). Lojistik regresyon, bir veri noktasının belirli bir sınıfa ait olma **olasılığını** hesaplar.

### 1.2. Sigmoid (Lojistik) Fonksiyonu

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Bu fonksiyon herhangi bir gerçel sayıyı 0 ile 1 arasına sıkıştırır:
- $z \to +\infty \Rightarrow \sigma(z) \to 1$
- $z \to -\infty \Rightarrow \sigma(z) \to 0$
- $z = 0 \Rightarrow \sigma(z) = 0.5$

### 1.3. Karar Eşiği (Decision Threshold)
Eğer hesaplanan olasılık $\ge 0.5$ ise model çıktıyı **1**, $< 0.5$ ise **0** olarak sınıflandırır.

### 1.4. Temel Terimler
1. **Sigmoid Fonksiyonu**: Olasılık çıktısı üretmek için kullanılır
2. **Log Loss (Maliyet Fonksiyonu)**: Modelin hatalarını cezalandıran fonksiyon
3. **Confusion Matrix**: Sınıflandırma başarısını gösteren matris
4. **Gradient Descent**: Ağırlıkları optimize etmek için kullanılır
5. **AUC-ROC**: Modelin tüm eşik değerlerindeki performansını ölçer

---
## 2. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

print("Kutuphaneler basariyla yuklendi.")

---
## 3. Veri Setinin Yüklenmesi ve Ön İşleme

Hocamızın orijinal notebook'unda kalp hastalığı verisi (`heart.csv`) kullanılmıştı. Biz aynı yapıyı koruyarak, finans/pazarlama dünyasından çok popüler bir alternatif olan **Bank Marketing** veri setini kullanacağız.

- **Hedef Değişken (Target):** `deposit` (Müşteri vadeli hesaba abone oldu mu? 1: Evet, 0: Hayır)
- **Özellikler (Features):** Yaş, yıllık ortalama bakiye, kampanya süresince yapılan görüşme sayısı vb. sayısal özellikler.

In [ ]:
# Veri setini doğrudan güvenilir bir GitHub raw bağlantısından çekiyoruz
url = "https://raw.githubusercontent.com/mdsheikhchal/Bank-Marketing-Term-Deposit-Prediction/master/bank.csv"
df_raw = pd.read_csv(url)

# Orijinal şablona tam uyum sağlaması için sadece sayısal sütunları ve hedef değişkeni seçelim
df = df_raw[['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'deposit']].copy()

# Hedef değişkeni 'yes'/'no' formatından 1/0 formatına dönüştürüyoruz
df['deposit'] = df['deposit'].map({'yes': 1, 'no': 0})

print("Veri Seti İlk 5 Satır:")
display(df.head())

print(f"\nVeri Seti Boyutu: {df.shape}")

### Eksik Değer Kontrolü

In [ ]:
print("Eksik Değer Sayıları:")
print(df.isnull().sum())

---
## 4. Özellik ve Hedef Değişken Ayrımı

In [ ]:
X = df.drop('deposit', axis=1)
y = df['deposit']

print("Özellikler (X) matris boyutu:", X.shape)
print("Hedef Değişken (y) vektör boyutu:", y.shape)

---
## 5. Verinin Eğitim ve Test Olarak Ayrılması

Veriyi orijinal koddaki gibi %80 Eğitim, %20 Test olacak şekilde ayırıyoruz.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Eğitim seti: {X_train.shape}")
print(f"Test seti: {X_test.shape}")

---
## 6. Verilerin Ölçeklendirilmesi (Feature Scaling)

Lojistik Regresyon katsayı analizi yaptığı ve arka planda Gradient Descent kullandığı için sayısal özelliklerin `StandardScaler` ile aynı ölçeğe getirilmesi kritik önem taşır.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Veriler başarıyla ölçeklendirildi.")

---
## 7. Modelin Oluşturulması ve Eğitilmesi

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("Lojistik Regresyon modeli başarıyla eğitildi.")

---
## 8. Modelin Tahminleri ve Performans Analizi

In [ ]:
# Sınıf tahminleri ve olasılık tahminleri
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Metriklerin hesaplanması
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

### Karmaşıklık Matrisi (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Abone Olmadı', 'Abone Oldu'], 
            yticklabels=['Abone Olmadı', 'Abone Oldu'])
plt.title('Confusion Matrix - Banka Kampanya Başarısı')
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.show()

### ROC Eğrisi (ROC Curve)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Eğrisi (AUC = {auc_roc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Yanlış Pozitif Oranı (False Positive Rate)')
plt.ylabel('Doğru Pozitif Oranı (True Positive Rate)')
plt.title('ROC Eğrisi Grafiği')
plt.legend(loc="lower right")
plt.show()

---
## 9. Model Sonuç Tablosu

In [ ]:
print("=== SONUÇ TABLOSU ===")
print(f"  Doğruluk (Accuracy):     {accuracy:.4f}  ({(accuracy*100):.2f}%)")
print(f"  Hassasiyet (Precision):  {precision:.4f}  ({(precision*100):.2f}%)")
print(f"  Duyarlılık (Recall):     {recall:.4f}  ({(recall*100):.2f}%)")
print(f"  F1-Skoru:                {f1:.4f}")
print(f"  AUC-ROC:                 {auc_roc:.4f}")